# Data Retrieval -- Tripadvisor Scraping

Note: I do not recommend re-running this notebook as it scrapes a lot of data, which takes a little while and is locally saved as part of the notebook.

In [57]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import undetected_chromedriver as uc
import time
import random
from bs4 import BeautifulSoup
import pandas as pd
import os

Scraping Tripadvisor requires the use of an undected chromedriver which needs to be set up first before it can be used to scrape the relevant URLs.

In [58]:
options = Options()
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

In [60]:
driver = uc.Chrome()
driver.get("https://www.tripadvisor.com")
time.sleep(random.uniform(4, 7))
driver.get("https://www.tripadvisor.com/Attractions-g189952-Activities-a_allAttractions.true-Iceland.html")

First check if things are working correctly.

In [21]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Check if the page loaded properly
print("Page title:", soup.title.string if soup.title else "No title found")
print("Total divs:", len(soup.find_all('div')))

# Try to find the review count automation tag directly
review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
print("Review count tags found:", len(review_tags))

rating_tags = soup.find_all(attrs={'data-automation': 'bubbleRatingValue'})
print("Rating tags found:", len(rating_tags))

# Check if tKstk class exists at all
cards = soup.find_all('div', class_='tKstk')
print("tKstk cards found:", len(cards))

Page title: THE 15 BEST Things to Do in Iceland (2026) - Must-See Attractions
Total divs: 2845
Review count tags found: 30
Rating tags found: 30
tKstk cards found: 0


I like to work in the way where I extract the full html file for each page, and then extract relevant information in order to have the scrape the page as little times as possible and have space to figure out the best way to extract the needed information afterwards.

In [34]:
os.makedirs('tripadvisor_pages', exist_ok=True)

base_url = "https://www.tripadvisor.com/Attractions-g189952-Activities-a_allAttractions.true-oa{}-Iceland.html"

for offset in range(0, 131 * 30, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    # Skip if already downloaded
    if os.path.exists(filepath):
        print(f"Offset {offset} already saved, skipping.")
        continue
    
    print(f"Scraping offset {offset}...")
    driver.get(base_url.format(offset))
    time.sleep(random.uniform(3, 6))
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(driver.page_source)
    
    print(f"  Saved to {filepath}")

print("Done.")

Scraping offset 0...
  Saved to tripadvisor_pages/page_0.html
Scraping offset 30...
  Saved to tripadvisor_pages/page_30.html
Scraping offset 60...
  Saved to tripadvisor_pages/page_60.html
Scraping offset 90...
  Saved to tripadvisor_pages/page_90.html
Scraping offset 120...
  Saved to tripadvisor_pages/page_120.html
Scraping offset 150...
  Saved to tripadvisor_pages/page_150.html
Scraping offset 180...
  Saved to tripadvisor_pages/page_180.html
Scraping offset 210...
  Saved to tripadvisor_pages/page_210.html
Scraping offset 240...
  Saved to tripadvisor_pages/page_240.html
Scraping offset 270...
  Saved to tripadvisor_pages/page_270.html
Scraping offset 300...
  Saved to tripadvisor_pages/page_300.html
Scraping offset 330...
  Saved to tripadvisor_pages/page_330.html
Scraping offset 360...
  Saved to tripadvisor_pages/page_360.html
Scraping offset 390...
  Saved to tripadvisor_pages/page_390.html
Scraping offset 420...
  Saved to tripadvisor_pages/page_420.html
Scraping offset 450.

In [39]:
files = os.listdir('tripadvisor_pages')
print(f"Total files: {len(files)}")
print(files[:10])

Total files: 134
['page_3630.html', 'page_2430.html', 'page_3870.html', 'page_2220.html', 'page_2670.html', 'page_1230.html', 'page_930.html', 'page_2580.html', 'page_510.html', 'page_1590.html']


Having successfully scraped the pages and saved them as html files, I can now extract the information I am interested in.

In [48]:
all_attractions = []

for offset in range(0, 97 * 30, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    if not os.path.exists(filepath):
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
    
    for review_tag in review_tags:
        # Name - strip ranking number
        name_tag = review_tag.find_previous('h3')
        if name_tag:
            rank_span = name_tag.find('span', class_='vAUKO')
            if rank_span:
                rank_span.decompose()
            name = name_tag.get_text(strip=True)
        else:
            name = None
        
        # Link
        link_tag = review_tag.find_previous('a', href=lambda h: h and 'Attraction_Review' in h)
        link = 'https://www.tripadvisor.com' + link_tag['href'] if link_tag else None
        
        # Rating
        rating_tag = review_tag.find_previous(attrs={'data-automation': 'bubbleRatingValue'})
        rating = rating_tag.get_text(strip=True) if rating_tag else None
        
        # Review count
        review_count = review_tag.get_text(strip=True).strip('()').replace(',', '')
        
        all_attractions.append({
            'name': name,
            'link': link,
            'rating': rating,
            'review_count': review_count,
        })

df_tripadvisor = pd.DataFrame(all_attractions)
print(f"Total attractions extracted: {len(df_tripadvisor)}")
print(df_tripadvisor.head(10))

Total attractions extracted: 2179
                                       name  \
0                           Hallgrimskirkja   
1                                    Perlan   
2                            Glacier Lagoon   
3                            Gullfoss Falls   
4                                Sky Lagoon   
5                                 Skogafoss   
6                               Blue Lagoon   
7  Harpa Concert Hall and Conference Centre   
8                 Thingvellir National Park   
9                  Seljalandsfoss waterfall   

                                                link rating review_count  
0  https://www.tripadvisor.com/Attraction_Review-...    4.4        23266  
1  https://www.tripadvisor.com/Attraction_Review-...    4.5         4305  
2  https://www.tripadvisor.com/Attraction_Review-...    4.9         5082  
3  https://www.tripadvisor.com/Attraction_Review-...    4.7        12522  
4  https://www.tripadvisor.com/Attraction_Review-...    4.5         4964 

Next, I want to check how many attractions I retrieved for each html page I scraped.

In [55]:
for offset in range(0, 131 * 30, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    if not os.path.exists(filepath):
        print(f"Offset {offset}: missing")
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
    print(f"Offset {offset}: {len(review_tags)} attractions")

Offset 0: 30 attractions
Offset 30: 30 attractions
Offset 60: 30 attractions
Offset 90: 30 attractions
Offset 120: 30 attractions
Offset 150: 30 attractions
Offset 180: 30 attractions
Offset 210: 30 attractions
Offset 240: 30 attractions
Offset 270: 30 attractions
Offset 300: 30 attractions
Offset 330: 30 attractions
Offset 360: 30 attractions
Offset 390: 30 attractions
Offset 420: 30 attractions
Offset 450: 30 attractions
Offset 480: 30 attractions
Offset 510: 29 attractions
Offset 540: 30 attractions
Offset 570: 29 attractions
Offset 600: 30 attractions
Offset 630: 30 attractions
Offset 660: 30 attractions
Offset 690: 30 attractions
Offset 720: 30 attractions
Offset 750: 30 attractions
Offset 780: 30 attractions
Offset 810: 30 attractions
Offset 840: 30 attractions
Offset 870: 30 attractions
Offset 900: 30 attractions
Offset 930: 30 attractions
Offset 960: 30 attractions
Offset 990: 30 attractions
Offset 1020: 30 attractions
Offset 1050: 30 attractions
Offset 1080: 30 attractions
Off

I need to check up on some pages that have no attractions.

In [50]:
with open('tripadvisor_pages/page_2220.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

print("Title:", soup.title.string if soup.title else "No title")
print("Total divs:", len(soup.find_all('div')))
print("bubbleReviewCount tags:", len(soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})))
print("\nFirst 1000 chars of body text:")
print(soup.get_text()[:1000])

Title: THE 15 BEST Things to Do in Iceland (2026) - Must-See Attractions
Total divs: 1622
bubbleReviewCount tags: 0

First 1000 chars of body text:
THE 15 BEST Things to Do in Iceland (2026) - Must-See AttractionsSkip to main contentPlan with AIRewardsDiscoverReviewUSDSign inIcelandHotelsThings to DoRestaurantsCruisesForumsEuropeIcelandThings to Do in IcelandTop Things to Do in IcelandTop Things to Do in Iceland - Iceland AttractionsTop Things to Do in Iceland Enter datesAttractionsFiltersSortAll things to doCategory typesAttractionsToursDay TripsOutdoor ActivitiesConcerts & ShowsFood & DrinkEventsClasses & WorkshopsShoppingTransportationTraveler ResourcesShow moreTypes of AttractionsNature & ParksSights & LandmarksMuseumsBoat Tours & Water SportsFun & GamesNightlifeSpas & WellnessClasses & WorkshopsWater & Amusement ParksZoos & AquariumsCasinos & GamblingShow moreAwardsTravelers' Choice Awards winners (including the "Best of the Best" title) are among the top 10% of listings on Tripad

In [62]:
# Find and delete files with 0 review tags
deleted = []
for offset in range(0, 97 * 30, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    if not os.path.exists(filepath):
        continue
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
    if len(review_tags) == 0:
        os.remove(filepath)
        deleted.append(offset)

print(f"Deleted {len(deleted)} empty files:")
print(deleted)

Deleted 23 empty files:
[2220, 2250, 2280, 2310, 2340, 2370, 2400, 2430, 2460, 2490, 2520, 2550, 2580, 2610, 2640, 2670, 2700, 2730, 2760, 2790, 2820, 2850, 2880]


In [53]:
os.remove('tripadvisor_pages/page_2190.html')

As some pages have no attractions, I will try to scrape them again to check if the issue is with the website blocking information from me.

In [63]:
base_url = "https://www.tripadvisor.com/Attractions-g189952-Activities-a_allAttractions.true-oa{}-Iceland.html"

for offset in range(2190, 2970, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    if os.path.exists(filepath):
        print(f"Offset {offset} already saved, skipping.")
        continue
    
    print(f"Scraping offset {offset}...")
    driver.get(base_url.format(offset))
    time.sleep(random.uniform(5, 10))
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(driver.page_source)
    
    print(f"  Saved to {filepath}")

print("Done.")

Offset 2190 already saved, skipping.
Scraping offset 2220...
  Saved to tripadvisor_pages/page_2220.html
Scraping offset 2250...
  Saved to tripadvisor_pages/page_2250.html
Scraping offset 2280...
  Saved to tripadvisor_pages/page_2280.html
Scraping offset 2310...
  Saved to tripadvisor_pages/page_2310.html
Scraping offset 2340...
  Saved to tripadvisor_pages/page_2340.html
Scraping offset 2370...
  Saved to tripadvisor_pages/page_2370.html
Scraping offset 2400...
  Saved to tripadvisor_pages/page_2400.html
Scraping offset 2430...
  Saved to tripadvisor_pages/page_2430.html
Scraping offset 2460...
  Saved to tripadvisor_pages/page_2460.html
Scraping offset 2490...
  Saved to tripadvisor_pages/page_2490.html
Scraping offset 2520...
  Saved to tripadvisor_pages/page_2520.html
Scraping offset 2550...
  Saved to tripadvisor_pages/page_2550.html
Scraping offset 2580...
  Saved to tripadvisor_pages/page_2580.html
Scraping offset 2610...
  Saved to tripadvisor_pages/page_2610.html
Scraping of

Again, I will check up on the attractions on these pages.

In [64]:
for offset in range(2190, 2970, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    if not os.path.exists(filepath):
        print(f"Offset {offset}: missing")
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
    print(f"Offset {offset}: {len(review_tags)} attractions")

Offset 2190: 1 attractions
Offset 2220: 0 attractions
Offset 2250: 0 attractions
Offset 2280: 0 attractions
Offset 2310: 0 attractions
Offset 2340: 0 attractions
Offset 2370: 0 attractions
Offset 2400: 0 attractions
Offset 2430: 0 attractions
Offset 2460: 0 attractions
Offset 2490: 0 attractions
Offset 2520: 0 attractions
Offset 2550: 0 attractions
Offset 2580: 0 attractions
Offset 2610: 0 attractions
Offset 2640: 0 attractions
Offset 2670: 0 attractions
Offset 2700: 0 attractions
Offset 2730: 0 attractions
Offset 2760: 0 attractions
Offset 2790: 0 attractions
Offset 2820: 0 attractions
Offset 2850: 0 attractions
Offset 2880: 0 attractions
Offset 2910: 0 attractions
Offset 2940: 0 attractions


There are still no attractions on these pages. However, these are also attractions in the 2000s of the list. I therefore decided I can live without them.